# CIC-IDS-2017 feature preparation

This notebook starts from the cleaned dataset produced by `01_data_exploration.ipynb`. Its first stage removes identifier columns that would encourage memorization of the CIC-IDS-2017 laboratory, separates the remaining features from the multiclass target, and creates a reproducible stratified train/test split. No scaling, encoding, correlation filtering, resampling or learned feature selection is performed before the split.

## 1. Imports and paths

In [1]:
from itertools import combinations
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

processed_dataset_candidates = [
    Path("../data/processed/cicids2017_cleaned.parquet"),
    Path("ml/data/processed/cicids2017_cleaned.parquet"),
]

PROCESSED_DATA_PATH = next(
    (
        path.resolve()
        for path in processed_dataset_candidates
        if path.exists()
    ),
    None,
)

if PROCESSED_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cicids2017_cleaned.parquet. "
        "Run 01_data_exploration.ipynb first."
    )

RANDOM_STATE = 42
TEST_SIZE = 0.20

print(f"Processed dataset: {PROCESSED_DATA_PATH}")
print(f"File size: {PROCESSED_DATA_PATH.stat().st_size / (1024 ** 2):,.2f} MiB")
print(f"Random state: {RANDOM_STATE}")
print(f"Test fraction: {TEST_SIZE:.0%}")

Processed dataset: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System\ml\data\processed\cicids2017_cleaned.parquet
File size: 360.48 MiB
Random state: 42
Test fraction: 20%


## 2. Load and validate the cleaned data

Confirm that the processed file matches the final dimensions and target structure recorded in the exploration notebook.

In [2]:
data = pd.read_parquet(PROCESSED_DATA_PATH)

EXPECTED_ROWS = 2_824_752
EXPECTED_COLUMNS = 76
EXPECTED_LABELS = 15

dataset_validation = pd.DataFrame(
    {
        "Observed": [
            data.shape[0],
            data.shape[1],
            data["Label"].nunique(dropna=False)
            if "Label" in data.columns
            else 0,
            int(data["Label"].isna().sum())
            if "Label" in data.columns
            else data.shape[0],
        ],
        "Expected": [
            EXPECTED_ROWS,
            EXPECTED_COLUMNS,
            EXPECTED_LABELS,
            0,
        ],
    },
    index=[
        "Rows",
        "Columns",
        "Detailed target classes",
        "Missing target labels",
    ],
)
display(dataset_validation)

if not dataset_validation["Observed"].equals(
    dataset_validation["Expected"]
):
    raise AssertionError(
        "The cleaned dataset does not match the expected EDA output."
    )

print("The cleaned dataset matches the final EDA output.")

,Observed,Expected
Rows,2824752,2824752
Columns,76,76
Detailed target classes,15,15
Missing target labels,0,0


The cleaned dataset matches the final EDA output.


## 3. Review all cleaned columns

Display the complete cleaned schema before excluding identifiers or separating the target.

In [3]:
cleaned_column_overview = pd.DataFrame(
    {
        "Position": range(1, len(data.columns) + 1),
        "Column": data.columns,
        "Data type": data.dtypes.astype(str).to_numpy(),
    }
).set_index("Position")

with pd.option_context("display.max_rows", None):
    display(cleaned_column_overview)
print(f"Total cleaned columns: {len(cleaned_column_overview)}")

,Column,Data type
Position,,
1,Flow ID,object
2,Src IP,object
3,Src Port,float64
4,Dst IP,object
5,Dst Port,float64
6,Protocol,float64
7,Timestamp,object
8,Flow Duration,float64
9,Total Fwd Packet,float64


Total cleaned columns: 76


## 4. Remove identifier and leakage-prone columns

`Flow ID`, endpoint IP addresses and `Timestamp` describe this particular laboratory capture rather than transferable flow behavior. These fixed schema exclusions do not depend on statistics calculated from the dataset, so remove them directly from the cleaned table before separating features and target. Ports and `Protocol` are retained for now.

In [4]:
identifier_columns = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp",
]
missing_identifier_columns = [
    column for column in identifier_columns if column not in data.columns
]
if missing_identifier_columns:
    raise KeyError(
        "Expected identifier columns are missing: "
        f"{missing_identifier_columns}"
    )

identifier_decisions = pd.DataFrame(
    {
        "Column": identifier_columns,
        "Reason for exclusion": [
            "Constructed flow identifier; encourages record memorization",
            "Source identity is specific to the CIC-IDS-2017 laboratory",
            "Destination identity is specific to the CIC-IDS-2017 laboratory",
            "Encodes the published attack schedule and capture day",
        ],
    }
)
display(identifier_decisions)

data.drop(columns=identifier_columns, inplace=True)
print(f"Columns remaining after fixed exclusions: {data.shape[1]}")

,Column,Reason for exclusion
0,Flow ID,Constructed flow identifier; encourages record...
1,Src IP,Source identity is specific to the CIC-IDS-201...
2,Dst IP,Destination identity is specific to the CIC-ID...
3,Timestamp,Encodes the published attack schedule and capt...


Columns remaining after fixed exclusions: 72


## 5. Separate features and target

Keep the original 15-class `Label` as the target. Using `pop` separates it without duplicating the large feature table in memory.

In [5]:
y = data.pop("Label")
X = data
del data

print(f"Feature rows: {X.shape[0]:,}")
print(f"Feature columns: {X.shape[1]}")
print(f"Target rows: {len(y):,}")
print(f"Target classes: {y.nunique()}")

Feature rows: 2,824,752
Feature columns: 71
Target rows: 2,824,752
Target classes: 15


## 6. Create the train/test split

Use a reproducible stratified 80/20 split so every target class is represented in both subsets. This split estimates performance on held-out flows from the same CIC-IDS-2017 capture; it does not by itself demonstrate generalization to a different network or time period. All later data-dependent preprocessing must be fitted using `X_train` and `y_train` only.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

if not X_train.index.intersection(X_test.index).empty:
    raise AssertionError("Training and test rows overlap.")

del X, y

split_size_summary = pd.DataFrame(
    {
        "Rows": [len(X_train), len(X_test)],
        "Percentage of dataset": [
            len(X_train) / (len(X_train) + len(X_test)) * 100,
            len(X_test) / (len(X_train) + len(X_test)) * 100,
        ],
    },
    index=["Training set", "Test set"],
)
display(split_size_summary)
print(f"Features in each split: {X_train.shape[1]}")

,Rows,Percentage of dataset
Training set,2259801,79.999979
Test set,564951,20.000021


Features in each split: 71


## 7. Verify split distributions

Confirm that no class disappeared and quantify how closely stratification preserved the original multiclass distribution.

In [7]:
training_label_counts = y_train.value_counts()
test_label_counts = y_test.value_counts()
total_label_counts = (
    training_label_counts.add(test_label_counts, fill_value=0)
    .astype("int64")
    .sort_values(ascending=False)
)

split_label_distribution = pd.DataFrame(
    {
        "Total flows": total_label_counts,
        "Training flows": training_label_counts.reindex(
            total_label_counts.index, fill_value=0
        ),
        "Test flows": test_label_counts.reindex(
            total_label_counts.index, fill_value=0
        ),
    }
)
split_label_distribution["Training percentage"] = (
    split_label_distribution["Training flows"] / len(y_train) * 100
)
split_label_distribution["Test percentage"] = (
    split_label_distribution["Test flows"] / len(y_test) * 100
)
split_label_distribution["Absolute difference (percentage points)"] = (
    split_label_distribution["Training percentage"]
    .sub(split_label_distribution["Test percentage"])
    .abs()
)
display(split_label_distribution)

if split_label_distribution[["Training flows", "Test flows"]].eq(0).any().any():
    raise AssertionError("At least one target class is absent from a split.")

if not (
    split_label_distribution["Training flows"]
    + split_label_distribution["Test flows"]
).equals(split_label_distribution["Total flows"]):
    raise AssertionError("Split label counts do not reproduce the full target.")

print("All 15 classes are present in both splits.")
print("The train/test split is complete; no preprocessing has been fitted.")

,Total flows,Training flows,Test flows,Training percentage,Test percentage,Absolute difference (percentage points)
Label,,,,,,
BENIGN,2268391,1814712,453679,80.304062,80.304133,7.039894e-05
DoS Hulk,229964,183971,45993,8.141027,8.141060,3.344402e-05
PortScan,158804,127043,31761,5.621867,5.621903,3.678832e-05
DDoS,128006,102405,25601,4.531594,4.531543,5.026754e-05
DoS GoldenEye,10288,8230,2058,0.364191,0.364279,8.801977e-05
FTP-Patator,7931,6345,1586,0.280777,0.280732,4.462437e-05
SSH-Patator,5895,4716,1179,0.208691,0.208691,2.770474e-07
DoS slowloris,5796,4637,1159,0.205195,0.205151,4.452403e-05
DoS Slowhttptest,5499,4399,1100,0.194663,0.194707,4.399320e-05


All 15 classes are present in both splits.
The train/test split is complete; no preprocessing has been fitted.


## 8. Calculate feature-selection diagnostics

Calculate feature-to-label mutual information and every unique Pearson and Spearman feature pair using only the training data. `Protocol` remains numeric in the data but is treated as discrete by the MI estimator and excluded from Pearson and Spearman because its numeric codes do not represent ordered quantities. Display the MI ranking and one combined correlation table instead of repeating separate Pearson and Spearman outputs.

In [8]:
TOP_CORRELATION_PAIRS_TO_DISPLAY = 100

mi_feature_columns = X_train.columns.tolist()
mi_discrete_feature_mask = [
    column == "Protocol" for column in mi_feature_columns
]

mi_scores = mutual_info_classif(
    X_train[mi_feature_columns],
    y_train,
    discrete_features=mi_discrete_feature_mask,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

mutual_information_ranking = pd.DataFrame(
    {
        "Feature": mi_feature_columns,
        "Mutual information": mi_scores,
        "MI estimator input type": [
            "Discrete" if is_discrete else "Continuous"
            for is_discrete in mi_discrete_feature_mask
        ],
    }
).sort_values(
    "Mutual information", ascending=False, ignore_index=True
)
mutual_information_ranking.index = pd.RangeIndex(
    start=1,
    stop=len(mutual_information_ranking) + 1,
    name="Rank",
)

correlation_feature_columns = [
    column for column in X_train.columns if column != "Protocol"
]
pearson_correlation_matrix = X_train[
    correlation_feature_columns
].corr(method="pearson")
spearman_correlation_matrix = X_train[
    correlation_feature_columns
].corr(method="spearman")

combined_filter_results = pd.DataFrame(
    (
        (
            first_feature,
            second_feature,
            pearson_correlation_matrix.at[
                first_feature, second_feature
            ],
            spearman_correlation_matrix.at[
                first_feature, second_feature
            ],
        )
        for first_feature, second_feature in combinations(
            correlation_feature_columns, 2
        )
    ),
    columns=[
        "Feature 1",
        "Feature 2",
        "Pearson correlation",
        "Spearman correlation",
    ],
)

mi_score_by_feature = mutual_information_ranking.set_index(
    "Feature"
)["Mutual information"]
combined_filter_results.insert(
    1,
    "Feature 1 MI",
    combined_filter_results["Feature 1"].map(mi_score_by_feature),
)
combined_filter_results.insert(
    3,
    "Feature 2 MI",
    combined_filter_results["Feature 2"].map(mi_score_by_feature),
)
combined_filter_results["Absolute Pearson correlation"] = (
    combined_filter_results["Pearson correlation"].abs()
)
combined_filter_results["Absolute Spearman correlation"] = (
    combined_filter_results["Spearman correlation"].abs()
)
combined_filter_results["Maximum absolute correlation"] = (
    combined_filter_results[
        [
            "Absolute Pearson correlation",
            "Absolute Spearman correlation",
        ]
    ].max(axis=1)
)
combined_filter_results.sort_values(
    "Maximum absolute correlation",
    ascending=False,
    inplace=True,
)
combined_filter_results.index = pd.RangeIndex(
    start=1,
    stop=len(combined_filter_results) + 1,
    name="Pair",
)

combined_filter_display = combined_filter_results.head(
    TOP_CORRELATION_PAIRS_TO_DISPLAY
).copy()
numeric_result_columns = [
    column
    for column in combined_filter_display.columns
    if column not in ["Feature 1", "Feature 2"]
]
combined_filter_display[numeric_result_columns] = (
    combined_filter_display[numeric_result_columns].round(6)
)

print(f"MI features scored: {len(mutual_information_ranking)}")
print(f"Correlation features included: {len(correlation_feature_columns)}")
print(f"Unique correlation pairs checked: {len(combined_filter_results):,}")
print("Mutual-information ranking:")
with pd.option_context("display.max_rows", None):
    display(mutual_information_ranking.round(6))

print(
    f"Top {TOP_CORRELATION_PAIRS_TO_DISPLAY} combined correlation pairs:"
)
with pd.option_context("display.max_rows", None):
    display(combined_filter_display)

MI features scored: 71
Correlation features included: 70
Unique correlation pairs checked: 2,415
Mutual-information ranking:


,Feature,Mutual information,MI estimator input type
Rank,,,
1,Average Packet Size,0.589162,Continuous
2,Packet Length Mean,0.561130,Continuous
3,Packet Length Std,0.558339,Continuous
4,Packet Length Variance,0.556995,Continuous
5,Subflow Bwd Bytes,0.498750,Continuous
6,Total Length of Bwd Packet,0.498369,Continuous
7,FWD Init Win Bytes,0.497415,Continuous
8,Bwd Packet Length Mean,0.490913,Continuous
9,Bwd Segment Size Avg,0.490572,Continuous


Top 100 combined correlation pairs:


,Feature 1,Feature 1 MI,Feature 2,Feature 2 MI,Pearson correlation,Spearman correlation,Absolute Pearson correlation,Absolute Spearman correlation,Maximum absolute correlation
Pair,,,,,,,,,
1,Bwd Packet Length Mean,0.490913,Bwd Segment Size Avg,0.490572,1.000000,1.000000,1.000000,1.000000,1.000000
2,Total Length of Bwd Packet,0.498369,Subflow Bwd Bytes,0.498750,1.000000,1.000000,1.000000,1.000000,1.000000
3,Total Length of Fwd Packet,0.485725,Subflow Fwd Bytes,0.485743,0.999999,1.000000,0.999999,1.000000,1.000000
4,Total Bwd packets,0.270560,Subflow Bwd Packets,0.270831,1.000000,1.000000,1.000000,1.000000,1.000000
5,Fwd URG Flags,0.000021,CWR Flag Count,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,Fwd PSH Flags,0.019300,SYN Flag Count,0.018975,1.000000,1.000000,1.000000,1.000000,1.000000
7,Total Fwd Packet,0.247153,Subflow Fwd Packets,0.246821,1.000000,1.000000,1.000000,1.000000,1.000000
8,Fwd Packet Length Mean,0.401005,Fwd Segment Size Avg,0.401476,1.000000,1.000000,1.000000,1.000000,1.000000
9,Packet Length Std,0.558339,Packet Length Variance,0.556995,0.924830,0.999984,0.924830,0.999984,0.999984


The calculations cover all 71 features for MI and all 2,415 unique pairs among the 70 non-`Protocol` features for correlation. MI is a univariate relevance diagnostic, while Pearson and Spearman identify linear and monotonic relationships between features. Neither measurement authorizes removal by itself: highly associated features can describe different network behavior, and globally low MI can overlook features that help detect rare classes. The combined output is therefore an audit used to support explicit decisions rather than an automatic threshold rule.

## 9. Document and validate manually reviewed redundancy decisions

Document the seven features selected for later removal after reviewing their definitions, observed values, mutual information and pairwise relationships. Validate that every feature and retained counterpart exists and attach the training-set correlation and MI evidence. This section records the decisions without modifying either data split.

In [9]:
features_to_remove = [
    "Fwd Segment Size Avg",
    "Bwd Segment Size Avg",
    "Subflow Fwd Packets",
    "Subflow Bwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Bwd Bytes",
    "Packet Length Variance",
]

feature_removal_decisions = pd.DataFrame(
    {
        "Remove": features_to_remove,
        "Retain instead": [
            "Fwd Packet Length Mean",
            "Bwd Packet Length Mean",
            "Total Fwd Packet",
            "Total Bwd packets",
            "Total Length of Fwd Packet",
            "Total Length of Bwd Packet",
            "Packet Length Std",
        ],
        "Reason": [
            "Same practical values apart from floating-point noise",
            "Same practical values apart from rounding noise",
            "Identical throughout the training data",
            "Identical throughout the training data",
            "Differs in only one training row",
            "Differs in only 16 training rows",
            "Near-monotonic transformed representation of standard deviation",
        ],
    }
)
missing_removal_features = [
    feature
    for feature in features_to_remove
    if feature not in X_train.columns
]
if missing_removal_features:
    raise KeyError(
        "Expected removal features are missing: "
        f"{missing_removal_features}"
    )

missing_retained_features = [
    feature
    for feature in feature_removal_decisions["Retain instead"]
    if feature not in X_train.columns
]
if missing_retained_features:
    raise KeyError(
        "Expected retained features are missing: "
        f"{missing_retained_features}"
    )

decision_evidence = []
for _, decision in feature_removal_decisions.iterrows():
    remove_feature = decision["Remove"]
    retain_feature = decision["Retain instead"]
    matching_relationship = combined_filter_results.loc[
        (
            (combined_filter_results["Feature 1"] == remove_feature)
            & (combined_filter_results["Feature 2"] == retain_feature)
        )
        | (
            (combined_filter_results["Feature 1"] == retain_feature)
            & (combined_filter_results["Feature 2"] == remove_feature)
        )
    ]
    if len(matching_relationship) != 1:
        raise AssertionError(
            "Expected one correlation relationship for "
            f"{remove_feature} and {retain_feature}."
        )

    relationship = matching_relationship.iloc[0]
    decision_evidence.append(
        {
            "Remove MI": mi_score_by_feature[remove_feature],
            "Retain MI": mi_score_by_feature[retain_feature],
            "Pearson correlation": relationship[
                "Pearson correlation"
            ],
            "Spearman correlation": relationship[
                "Spearman correlation"
            ],
        }
    )

feature_removal_decisions = pd.concat(
    [
        feature_removal_decisions,
        pd.DataFrame(decision_evidence),
    ],
    axis=1,
)
display(feature_removal_decisions.round(6))

print(f"Documented decisions: {len(feature_removal_decisions)}")
print("All removal and retained features were validated.")
print("X_train and X_test remain unchanged.")

,Remove,Retain instead,Reason,Remove MI,Retain MI,Pearson correlation,Spearman correlation
0,Fwd Segment Size Avg,Fwd Packet Length Mean,Same practical values apart from floating-poin...,0.401476,0.401005,1.000000,1.000000
1,Bwd Segment Size Avg,Bwd Packet Length Mean,Same practical values apart from rounding noise,0.490572,0.490913,1.000000,1.000000
2,Subflow Fwd Packets,Total Fwd Packet,Identical throughout the training data,0.246821,0.247153,1.000000,1.000000
3,Subflow Bwd Packets,Total Bwd packets,Identical throughout the training data,0.270831,0.270560,1.000000,1.000000
4,Subflow Fwd Bytes,Total Length of Fwd Packet,Differs in only one training row,0.485743,0.485725,0.999999,1.000000
5,Subflow Bwd Bytes,Total Length of Bwd Packet,Differs in only 16 training rows,0.498750,0.498369,1.000000,1.000000
6,Packet Length Variance,Packet Length Std,Near-monotonic transformed representation of s...,0.556995,0.558339,0.924830,0.999984


Documented decisions: 7
All removal and retained features were validated.
X_train and X_test remain unchanged.


All seven decisions are documented with their retained counterparts, reasons, MI scores and Pearson/Spearman relationships. The validation confirms that every referenced column and pair is present. No columns are removed here, and both `X_train` and `X_test` remain unchanged with 71 features.